# EPSS and CISA KEV Loading

This notebook loads 2025-2026 EPSS daily CSV files (filtered to CISA KEV catalog CVEs) and the CISA KEV catalog itself, both capped at `CUTOFF_DATE`, and exports both as parquet files for `epss_kev_matching-v5.ipynb` to consume without re-running the ~5 minute load.

In [ ]:
# 0. Define the analysis cutoff date -- caps which EPSS files (cell 1) and KEV
# entries (cell 2) are loaded, so results stay reproducible as new data arrives.

import pandas as pd
import gzip
import shutil
from pathlib import Path
import re

ROOT = Path.cwd()
EPSS_YEAR_DIRS = [ROOT / "2025", ROOT / "2026"]
CUTOFF_DATE = pd.Timestamp("2026-07-31")
print(f"Analysis cutoff date: {CUTOFF_DATE.date()}")

In [ ]:
# 1. Ensure CSV files exist locally (decompress from .csv.gz if missing)

for epss_year_dir in EPSS_YEAR_DIRS:
    gz_paths = sorted(epss_year_dir.glob("epss_scores-*.csv.gz"))
    decompressed_count = 0
    for gz_path in gz_paths:
        csv_path = gz_path.with_suffix("")  # strips only ".gz"
        if not csv_path.exists():
            with gzip.open(gz_path, "rb") as src, open(csv_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
            decompressed_count += 1

    csv_dates = {
        p.name.removeprefix("epss_scores-").removesuffix(".csv")
        for p in epss_year_dir.glob("epss_scores-*.csv")
    }
    gz_dates = {
        p.name.removeprefix("epss_scores-").removesuffix(".csv.gz")
        for p in gz_paths
    }
    # Compare the actual date SETS, not just counts -- equal counts (e.g. 365 == 365)
    # could still hide a date missing from one side and a different date duplicated.
    if csv_dates != gz_dates:
        raise ValueError(
            f"{epss_year_dir.name}: .csv and .csv.gz dates don't match -- "
            f"dates only in .csv: {sorted(csv_dates - gz_dates)}; "
            f"dates only in .csv.gz: {sorted(gz_dates - csv_dates)}"
        )
    print(
        f"{epss_year_dir.name}: decompressed {decompressed_count:,} new .csv files, "
        f"{len(csv_dates):,} .csv dates exactly match {len(gz_dates):,} .csv.gz dates"
    )

In [ ]:
# 2. Error check: no gaps in calendar-day file coverage
# Verifies every day from 2025-01-01 through CUTOFF_DATE has an EPSS file, so a
# day the upstream feed never published doesn't silently shrink the dataset.
# A short allowlist covers confirmed-natural gaps (the EPSS feed itself never
# published that day) so those don't block the notebook.

KNOWN_MISSING_EPSS_DATES = set()

expected_dates = pd.date_range("2025-01-01", CUTOFF_DATE, freq="D")

available_dates = {
    daily_path.name.removeprefix("epss_scores-").removesuffix(".csv")
    for epss_year_dir in EPSS_YEAR_DIRS
    for daily_path in epss_year_dir.glob("epss_scores-*.csv")
}

missing_dates = sorted(
    d.strftime("%Y-%m-%d")
    for d in expected_dates
    if d.strftime("%Y-%m-%d") not in available_dates
)
unexpected_missing_dates = [d for d in missing_dates if d not in KNOWN_MISSING_EPSS_DATES]

if unexpected_missing_dates:
    raise ValueError(
        f"{len(unexpected_missing_dates):,} calendar days between 2025-01-01 and "
        f"{CUTOFF_DATE.date()} have no EPSS file and aren't in KNOWN_MISSING_EPSS_DATES: "
        f"{unexpected_missing_dates}"
    )

known_gaps_hit = sorted(set(missing_dates) & KNOWN_MISSING_EPSS_DATES)
print(
    f"OK: all calendar days from 2025-01-01 to {CUTOFF_DATE.date()} have a file, "
    f"except {len(known_gaps_hit):,} known gap(s): {known_gaps_hit}"
)

In [ ]:
# 3. Error check: every EPSS file (through CUTOFF_DATE) has the expected metadata
# and header lines, and the score_date in its metadata matches its filename date.
# Cell 1 reads with skiprows=1, which is only correct if line 1 is the
# "#model_version:...,score_date:..." metadata line and line 2 is the column header.
# Reads just the first two lines of each file, so this is fast.


EPSS_METADATA_PATTERN = re.compile(
    r"^#model_version:(?P<model_version>[^,]+),score_date:(?P<score_date>\d{4}-\d{2}-\d{2})T"
)
EXPECTED_EPSS_HEADER = "cve,epss,percentile"

files_missing_metadata = []
files_with_wrong_header = []
files_with_mismatched_date = []
model_version_by_file_date = {}
checked_file_count = 0

for epss_year_dir in EPSS_YEAR_DIRS:
    for daily_path in sorted(epss_year_dir.glob("epss_scores-*.csv")):
        file_date = daily_path.name.removeprefix("epss_scores-").removesuffix(".csv")
        if pd.Timestamp(file_date) > CUTOFF_DATE:
            continue
        checked_file_count += 1
        with open(daily_path, "r", encoding="utf-8") as f:
            metadata_line = f.readline().strip()
            header_line = f.readline().strip()

        metadata_match = EPSS_METADATA_PATTERN.match(metadata_line)
        if not metadata_match:
            files_missing_metadata.append((daily_path.name, metadata_line[:80]))
            continue
        if header_line != EXPECTED_EPSS_HEADER:
            files_with_wrong_header.append((daily_path.name, header_line[:80]))
        if metadata_match["score_date"] != file_date:
            files_with_mismatched_date.append((daily_path.name, metadata_match["score_date"]))
        model_version_by_file_date[file_date] = metadata_match["model_version"]

if files_missing_metadata:
    raise ValueError(
        f"{len(files_missing_metadata):,} EPSS files don't start with a "
        f"#model_version/score_date metadata line (skiprows=1 would be wrong): "
        f"{files_missing_metadata[:10]}"
    )
if files_with_wrong_header:
    raise ValueError(
        f"{len(files_with_wrong_header):,} EPSS files don't have '{EXPECTED_EPSS_HEADER}' "
        f"as their second line: {files_with_wrong_header[:10]}"
    )
if files_with_mismatched_date:
    raise ValueError(
        f"{len(files_with_mismatched_date):,} EPSS files have a metadata score_date that "
        f"doesn't match the filename date (file, metadata date): {files_with_mismatched_date[:10]}"
    )

model_version_ranges = (
    pd.Series(model_version_by_file_date, name="model_version")
    .rename_axis("file_date")
    .reset_index()
    .groupby("model_version")["file_date"]
    .agg(first_date="min", last_date="max", files="count")
    .sort_values("first_date")
)

print(f"OK: {checked_file_count:,} EPSS files checked (through {CUTOFF_DATE.date()})")
print("  - every file starts with a #model_version/score_date metadata line")
print(f"  - every file's second line is '{EXPECTED_EPSS_HEADER}'")
print("  - every file's metadata score_date matches its filename date")
print(f"EPSS model versions by file date:\n{model_version_ranges.to_string()}")

In [ ]:
# 4. Load EPSS scores (through CUTOFF_DATE) into a pandas DataFrame, filtered to KEV catalog CVEs
KEV_PATH = ROOT / "known_exploited_vulnerabilities.csv"

# Read cveID + dateAdded here (not just cveID) so this filter list can mirror
# cell 2's own filters exactly (dateAdded <= CUTOFF_DATE, CVE-2025/2026
# IDs only). Without matching cell 2's cutoff, EpssDf would include CVEs added
# to KEV after CUTOFF_DATE -- their EPSS history is still real and in-window,
# but Kev (cutoff-scoped) wouldn't consider them KEV members yet, so the two
# exported frames would silently disagree on which CVEs are "in KEV".

KevRaw = pd.read_csv(KEV_PATH, usecols=["cveID", "dateAdded"], parse_dates=["dateAdded"])
KevRaw["dateAdded"] = KevRaw["dateAdded"].dt.normalize()
KevCveIdsForFilter = set(
    KevRaw.loc[
        (KevRaw["dateAdded"] <= CUTOFF_DATE)
        & KevRaw["cveID"].str.startswith(("CVE-2025-", "CVE-2026-")),
        "cveID",
    ].dropna()
)

# Print counts by CVE-ID year so a wrong/unexpected year is obvious on inspection.
filter_year_counts = (
    pd.Series(sorted(KevCveIdsForFilter))
    .str.extract(r"^CVE-(\d{4})-")[0].rename("cve_id_year")
    .value_counts()
    .sort_index()
)
print(f"KevCveIdsForFilter by CVE-ID year:\n{filter_year_counts.to_string()}")


# Scores are taken as published each day, from whichever EPSS model FIRST was running 
# at the time (v2023.03.01, v2025.03.14, v2026.06.15). 
# Twenty-four CVEs have a 30-day pre-KEV window that spans a model change.

daily_frames = []
daily_raw_row_counts = []
for epss_year_dir in EPSS_YEAR_DIRS:
    print(f"Starting {epss_year_dir.name}...")
    for daily_path in sorted(epss_year_dir.glob("epss_scores-*.csv")):
        date_str = daily_path.name.removeprefix("epss_scores-").removesuffix(".csv")
        score_date = pd.Timestamp(date_str)
        if score_date > CUTOFF_DATE:
            continue
        daily_scores = pd.read_csv(
            daily_path,
            skiprows=1,
            usecols=["cve", "epss", "percentile"],
        )
        daily_raw_row_counts.append(len(daily_scores))
        daily_scores = daily_scores.loc[daily_scores["cve"].isin(KevCveIdsForFilter)].copy()
        daily_scores["score_date"] = score_date
        daily_frames.append(daily_scores)
    print(f"Finished {epss_year_dir.name}")

if not daily_frames:
    raise FileNotFoundError(f"No epss_scores-*.csv files found under {EPSS_YEAR_DIRS}")

EpssDf = pd.concat(daily_frames, ignore_index=True)

# These are calendar dates with no time component, but pd.Timestamp(date_str)
# infers second resolution here, and a parquet round-trip would then shift that
# to millisecond -- silently diverging from Kev's dateAdded (datetime64[ns], via
# read_csv's date parser). Normalize to datetime64[ns] now so both exported
# frames agree on dtype and downstream merges don't need their own casts.

EpssDf["score_date"] = EpssDf["score_date"].astype("datetime64[ns]")

first_score_date = EpssDf["score_date"].min()
latest_score_date = EpssDf["score_date"].max()

print(f"EPSS daily files loaded (through {CUTOFF_DATE.date()}): {len(daily_frames):,}")
print(f"EPSS rows (KEV CVEs only): {len(EpssDf):,}")
print(f"EPSS columns: {list(EpssDf.columns)}")
print(f"EPSS date range: {first_score_date.date()} to {latest_score_date.date()}")
print(f"Latest EPSS score date: {latest_score_date.date()}")

epss_cve_year_counts = (
    pd.Series(EpssDf["cve"].unique())
    .str.extract(r"^CVE-(\d{4})-")[0].rename("cve_id_year")
    .value_counts()
    .sort_index()
)
print(f"EpssDf unique CVEs by CVE-ID year:\n{epss_cve_year_counts.to_string()}")

In [ ]:
# 5. Error check: independently re-verify cell 1's file set and record counts
# Re-globs the source directories and re-counts raw lines by hand (not pandas),
# so this doesn't just trust cell 1's own daily_frames/daily_raw_row_counts.

independent_daily_paths = []
independent_raw_row_counts = []

for epss_year_dir in EPSS_YEAR_DIRS:
    print(f"Starting {epss_year_dir.name}...")
    for daily_path in sorted(epss_year_dir.glob("epss_scores-*.csv")):
        if pd.Timestamp(daily_path.name.removeprefix("epss_scores-").removesuffix(".csv")) > CUTOFF_DATE:
            continue
        independent_daily_paths.append(daily_path)
        with open(daily_path, "r", encoding="utf-8") as f:
            line_count = sum(1 for _ in f)
        independent_raw_row_counts.append(line_count - 2)  # metadata line + header line
    print(f"Finished {epss_year_dir.name}")

assert len(independent_daily_paths) == len(daily_frames), (
    f"File count mismatch: cell 1 processed {len(daily_frames):,} files, but "
    f"re-scanning the same directories found {len(independent_daily_paths):,}"
)

independent_total_raw_records = sum(independent_raw_row_counts)
cell1_total_raw_records = sum(daily_raw_row_counts)

assert independent_total_raw_records == cell1_total_raw_records, (
    f"Raw record count mismatch: cell 1 counted {cell1_total_raw_records:,} rows via "
    f"pandas, but an independent line count found {independent_total_raw_records:,}"
)

total_filtered_records = sum(len(frame) for frame in daily_frames)
assert total_filtered_records == len(EpssDf), (
    f"Row count mismatch after concat: {total_filtered_records:,} KEV-filtered rows "
    f"across daily files vs {len(EpssDf):,} rows in EpssDf"
)

print(f"Files: {len(independent_daily_paths):,} (independently re-scanned, matches cell 1)")
print(f"Raw records: {independent_total_raw_records:,} (independent line count, matches cell 1's pandas count)")
print(f"KEV-filtered records: {total_filtered_records:,} (matches EpssDf's {len(EpssDf):,} rows)")
print("OK: cell 1's file set, record counts, and concat all independently verified.")

In [ ]:
# 6. Error check: EpssDf has one row per CVE per day, no blank values, and
# scores/percentiles within the valid 0-1 range

duplicate_cve_days = EpssDf.loc[
    EpssDf.duplicated(["cve", "score_date"], keep=False),
    ["cve", "score_date"],
].drop_duplicates()
if not duplicate_cve_days.empty:
    raise ValueError(
        f"{len(duplicate_cve_days):,} (cve, score_date) pairs appear more than once in EpssDf: "
        f"{duplicate_cve_days.head(10).to_string(index=False)}"
    )

blank_counts = EpssDf[["cve", "epss", "percentile", "score_date"]].isna().sum()
if blank_counts.any():
    raise ValueError(
        f"EpssDf has blank values:\n{blank_counts[blank_counts > 0].to_string()}"
    )

for value_col in ("epss", "percentile"):
    out_of_range = EpssDf.loc[~EpssDf[value_col].between(0.0, 1.0)]
    if not out_of_range.empty:
        raise ValueError(
            f"{len(out_of_range):,} EpssDf rows have {value_col} outside 0-1:\n"
            f"{out_of_range.head(10).to_string(index=False)}"
        )

print(f"OK: no duplicate (cve, score_date) pairs across {len(EpssDf):,} EpssDf rows")
print("OK: no blank cve, epss, percentile, or score_date values")
print(
    f"OK: epss within 0-1 (observed {EpssDf['epss'].min():.5f} to {EpssDf['epss'].max():.5f}); "
    f"percentile within 0-1 (observed {EpssDf['percentile'].min():.5f} to {EpssDf['percentile'].max():.5f})"
)

In [ ]:
# 7. Load the CISA KEV data (through CUTOFF_DATE, CVE-2025/2026 IDs only)

KEV_PATH = ROOT / "known_exploited_vulnerabilities.csv"

Kev = pd.read_csv(KEV_PATH, parse_dates=["dateAdded", "dueDate"])
Kev["dateAdded"] = Kev["dateAdded"].dt.normalize()

# A missing dateAdded would otherwise be silently dropped by the cutoff filter
# below (NaT <= CUTOFF_DATE is False), so check for it explicitly and halt.
KevMissingDateAdded = Kev.loc[Kev["dateAdded"].isna(), "cveID"]
if not KevMissingDateAdded.empty:
    raise ValueError(
        f"{len(KevMissingDateAdded):,} KEV entries have no dateAdded: "
        f"{sorted(KevMissingDateAdded.tolist())}"
    )

# The KEV catalog should have one row per CVE -- a duplicate would otherwise
# surface later as a cryptic validate="one_to_one" MergeError downstream.
KevDuplicateCveIds = Kev.loc[Kev["cveID"].duplicated(keep=False), "cveID"]
if not KevDuplicateCveIds.empty:
    raise ValueError(
        f"{KevDuplicateCveIds.nunique():,} duplicate cveID(s) in the KEV catalog: "
        f"{sorted(KevDuplicateCveIds.unique().tolist())}"
    )

Kev = Kev.loc[Kev["dateAdded"] <= CUTOFF_DATE].copy()

# Restrict to CVE-2025/2026 IDs, matching cell 1's KevCveIdsForFilter
# exactly, so EpssDf and Kev always agree on which CVEs are in scope.
Kev = Kev.loc[Kev["cveID"].str.startswith(("CVE-2025-", "CVE-2026-"))].copy()

print(f"KEV rows (dateAdded <= {CUTOFF_DATE.date()}, CVE-2025/2026 IDs only): {len(Kev):,}")
print(f"KEV unique CVEs: {Kev['cveID'].nunique():,}")
print(f"KEV date range: {Kev['dateAdded'].min().date()} to {Kev['dateAdded'].max().date()}")

# Print counts by CVE-ID year so a wrong/unexpected year is obvious on inspection.
kev_cve_year_counts = Kev["cveID"].str.extract(r"^CVE-(\d{4})-")[0].rename("cve_id_year").value_counts().sort_index()
print(f"Kev rows by CVE-ID year:\n{kev_cve_year_counts.to_string()}")
#Kev.head()

In [ ]:
# 8. Error check: every KEV CVE (through CUTOFF_DATE) must have at least one
# EPSS score somewhere in EpssDf -- it may be a low/zero score, but it must be
# present. A gap here means a KEV/EPSS CVE ID mismatch or missing coverage, and
# should stop the notebook rather than silently exporting incomplete data.

KevCvesMissingFromEpss = set(Kev["cveID"].dropna()) - set(EpssDf["cve"].unique())

if KevCvesMissingFromEpss:
    raise ValueError(
        f"{len(KevCvesMissingFromEpss):,} KEV CVEs (through {CUTOFF_DATE.date()}) have "
        f"no EPSS score at all in EpssDf: {sorted(KevCvesMissingFromEpss)}"
    )

print(
    f"OK: all {Kev['cveID'].nunique():,} KEV CVEs (through {CUTOFF_DATE.date()}) "
    "have at least one EPSS score in EpssDf."
)

In [ ]:
# 9. Export Kev and EpssDf (EPSS scores for KEV CVEs) to parquet for the plotting notebook

EXPORT_DIR = ROOT / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

Kev.to_parquet(EXPORT_DIR / "kev.parquet", index=False)
EpssDf.to_parquet(EXPORT_DIR / "epss_for_kev_cves.parquet", index=False)

print(f"Exported Kev: {len(Kev):,} rows -> {EXPORT_DIR / 'kev.parquet'}")
print(f"Exported EpssDf: {len(EpssDf):,} rows -> {EXPORT_DIR / 'epss_for_kev_cves.parquet'}")